In [1]:
import re
import pandas as pd
import os 

notebook_path= os.getcwd()
in_dir_notifications  = os.path.abspath(os.path.join(notebook_path,"..","..","Data","notification_historical","updated_notifications.jsonl"))
in_dir_master_direction = os.path.abspath(os.path.join(notebook_path,"..","..","Data","master_directory","master_directory.jsonl"))
circulars = pd.read_json(in_dir_notifications, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines = True)
# regexes to catch "Reserve Bank of India (X) ... Directions" and the no-paren variant
p1 = re.compile(r"Reserve Bank of India\s*[-–—]?\s*\(([^)]+)\)\s*(?:[A-Za-z]+\s+){0,3}Directions", re.IGNORECASE)
p2 = re.compile(r"Reserve Bank of India\s*[-–—]\s*([A-Za-z ,]+?),?\s*Directions", re.IGNORECASE)

norm = lambda s: re.sub(r"\s+", " ", s.replace("–","-").replace("—","-")).strip(" ,-").lower()

# extract candidate names from title + text
circulars["found_title"] = circulars["title"].apply(lambda t: [norm(x) for x in (p1.findall(t) + p2.findall(t))] if isinstance(t, str) else [])
circulars["found_text"]  = circulars["text"].apply(lambda t: [norm(x) for x in (p1.findall(t) + p2.findall(t))] if isinstance(t, str) else [])

# build master lookup: normalized name -> master direction id
master_lookup = {}
for _, r in master_directions.iterrows():
    for name in [norm(x) for x in (p1.findall(r["title"]) + p2.findall(r["title"]))]:
        master_lookup[name] = r["id"]

# match: prefer title hit, fall back to text hit
def _match(row):
    for n in row["found_title"]:
        if n in master_lookup: return master_lookup[n], "title_match"
    for n in row["found_text"]:
        if n in master_lookup: return master_lookup[n], "text_match"
    return None, "no_match",

circulars[["matched_id", "match_method"]] = circulars.apply(lambda r: pd.Series(_match(r)), axis=1)

# stats
print(circulars["match_method"].value_counts())
print(f"matched: {(circulars['match_method']!='no_match').mean():.1%}")

circulars[["id","title","found_title","matched_id","match_method"]]

match_method
no_match       97
title_match    34
text_match      7
Name: count, dtype: int64
matched: 29.7%


,id,title,found_title,matched_id,match_method
0,13167,Reserve Bank of India (Setting Up of Wholly Ow...,[],NaN,no_match
1,13168,Reserve Bank of India (Universal Banks – Licen...,[],NaN,no_match
2,13169,Compliance with Know Your Customer (KYC) norms,[],12943.0,text_match
3,13170,Consolidation of Regulations – Withdrawal of c...,[],NaN,no_match
4,13171,Compliance with Know Your Customer (KYC) norms,[],NaN,no_match
...,...,...,...,...,...
133,13675,Formation of new districts in the Union Territ...,[],NaN,no_match
134,13676,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match
135,13677,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match
136,13678,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match


In [2]:
no_match = circulars[circulars["match_method"] == "no_match"]
print(len(no_match))
for t in no_match["title"].head(10):
    print("-", t)
no_match["text"].iloc[0]

97
- Reserve Bank of India (Setting Up of Wholly Owned Subsidiaries by Foreign Banks) Guidelines, 2025 (Updated as on April 1, 2026)
- Reserve Bank of India (Universal Banks – Licensing) Guidelines, 2025
- Consolidation of Regulations – Withdrawal of circulars
- Compliance with Know Your Customer (KYC) norms
- Liberalised Remittance Scheme (LRS)- Submission of ‘LRS Daily Return’ by Authorised Dealers- Category -II banks/ entities and Full- Fledged Money Changers
- Liquidity Adjustment Facility - Change in rates
- Standing Liquidity Facility for Primary Dealers
- Penal Interest on shortfall in CRR and SLR requirements - Change in Bank Rate
- Reserve Bank of India (Non-Operative Financial Holding Company) (Amendment) Directions, 2025
- Export and Import of Indian Currency to or from Nepal and Bhutan


'RBI/DOR/2025-26/144\nNovember 28, 2025\nPrevious Versions\nReserve Bank of India (Setting Up of Wholly Owned Subsidiaries by Foreign Banks) Guidelines, 2025 (Updated as on April 1, 2026)\nA. Background\nThe global financial crisis of 2008 demonstrated that the growing complexity and interconnectedness of financial institutions, coupled with the lack of effective cross-border resolution regimes, severely constrained the ability of home and host authorities to cope with the failure of too big to fail (TBTF) and too connected to fail (TCTF) institutions. Globally, several policy options have been proposed to address these challenges, including measures to contain the negative externalities arising out of size and interconnectedness, strengthening the capital and liquidity buffers, and enhancing the resolvability of such institutions. The lessons from the global financial crisis support the case for domestic incorporation of foreign banks. The main advantages of local incorporation includ

In [14]:
circulars["footnotes"] = circulars["text"].apply(
    lambda t: [
        {
            "action": action.strip(),
            "effective_date": eff_date.strip(),
            "vide_ref": norm(vide_ref),
            "notification_date": notif_date.strip(),
        }
        for action, eff_date, vide_ref, notif_date in footnote_pattern.findall(t)
    ] if isinstance(t, str) else []
)

master_directions["footnotes"] = master_directions["text"].apply(
    lambda t: [
        {
            "action": action.strip(),
            "effective_date": eff_date.strip(),
            "vide_ref": norm(vide_ref),
            "notification_date": notif_date.strip(),
        }
        for action, eff_date, vide_ref, notif_date in footnote_pattern.findall(t)
    ] if isinstance(t, str) else []
)

print(f"circulars with footnotes: {(circulars['footnotes'].apply(len) > 0).sum()} / {len(circulars)}")
print(f"master_directions with footnotes: {(master_directions['footnotes'].apply(len) > 0).sum()} / {len(master_directions)}")

circulars[["id","title","footnotes"]]

circulars with footnotes: 1 / 138
master_directions with footnotes: 25 / 44


,id,title,footnotes
0,13167,Reserve Bank of India (Setting Up of Wholly Ow...,"[{'action': 'Substituted', 'effective_date': '..."
1,13168,Reserve Bank of India (Universal Banks – Licen...,[]
2,13169,Compliance with Know Your Customer (KYC) norms,[]
3,13170,Consolidation of Regulations – Withdrawal of c...,[]
4,13171,Compliance with Know Your Customer (KYC) norms,[]
...,...,...,...
133,13675,Formation of new districts in the Union Territ...,[]
134,13676,"Implementation of Section 51A of UAPA, 1967: U...",[]
135,13677,"Implementation of Section 51A of UAPA, 1967: U...",[]
136,13678,"Implementation of Section 51A of UAPA, 1967: U...",[]


In [8]:
import re
import pandas as pd
import os

notebook_path = os.getcwd()
in_dir_notifications_raw = os.path.abspath(os.path.join(notebook_path,"..","..","Data","notification_historical","notification.jsonl"))
in_dir_master_direction = os.path.abspath(os.path.join(notebook_path,"..","..","Data","master_directory","master_directory.jsonl"))

circulars = pd.read_json(in_dir_notifications_raw, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines=True)

# ── citation regexes ─────────────────────────────────────
DOC_TYPE = r"(?:Directions?|Guidelines?|Regulations?|Rules?|Circulars?|Framework|Scheme)"
p1 = re.compile(
    r"Reserve Bank of India\s*[-–—]?\s*\(([^)]+)\)"
    r"(?:\s*\([^)]*\))*"
    r"\s*(?:[A-Za-z]+\s+){0,3}" + DOC_TYPE,
    re.IGNORECASE
)
p2 = re.compile(r"Reserve Bank of India\s*[-–—]\s*([A-Za-z ,]+?),?\s*Directions", re.IGNORECASE)
norm = lambda s: re.sub(r"\s+", " ", s.replace("–","-").replace("—","-")).strip(" ,-").lower()

# ── extraction ────────────────────────────────────────────
circulars["found_title"] = circulars["title"].apply(lambda t: [norm(x) for x in (p1.findall(t)+p2.findall(t))] if isinstance(t, str) else [])
circulars["found_text"]  = circulars["text"].apply(lambda t: [norm(x) for x in (p1.findall(t)+p2.findall(t))] if isinstance(t, str) else [])

master_lookup = {}
for _, r in master_directions.iterrows():
    for name in [norm(x) for x in (p1.findall(r["title"])+p2.findall(r["title"]))]:
        master_lookup[name] = r["id"]

def _match(row):
    for n in row["found_title"]:
        if n in master_lookup: return master_lookup[n], "title_match"
    for n in row["found_text"]:
        if n in master_lookup: return master_lookup[n], "text_match"
    return None, "no_match"

circulars[["matched_id","match_method"]] = circulars.apply(lambda r: pd.Series(_match(r)), axis=1)

# ── three-bucket split ───────────────────────────────────
nbfc_pattern = r"non[\s-]?banking financial compan|nbfc"
circulars["is_nbfc_relevant"] = (
    circulars["text"].str.contains(nbfc_pattern, case=False, na=False) |
    circulars["title"].str.contains(nbfc_pattern, case=False, na=False)
)
circulars["has_citation"] = circulars["found_title"].apply(len).gt(0) | circulars["found_text"].apply(len).gt(0)

def bucket(row):
    if row["match_method"] != "no_match":
        return "linked"
    if not row["is_nbfc_relevant"]:
        return "not_nbfc"
    if not row["has_citation"]:
        return "nbfc_standalone"
    return "nbfc_citation_unmatched"

circulars["bucket"] = circulars.apply(bucket, axis=1)
print(circulars["bucket"].value_counts())

bucket
not_nbfc                   530
nbfc_citation_unmatched    134
linked                     102
Name: count, dtype: int64


In [9]:
for b in ["not_nbfc", "nbfc_standalone", "nbfc_citation_unmatched"]:
    print(f"\n=== {b} ===")
    print(circulars[circulars["bucket"]==b]["title"].head(10).to_string())


=== not_nbfc ===
41    Reserve Bank of India (All India Financial Ins...
43    Reserve Bank of India (All India Financial Ins...
48    Reserve Bank of India (All India Financial Ins...
49    Reserve Bank of India (All India Financial Ins...
57    Reserve Bank of India (All India Financial Ins...
60    Reserve Bank of India (Rural Co-operative Bank...
61    Reserve Bank of India (Rural Co-operative Bank...
62    Reserve Bank of India (Rural Co-operative Bank...
64    Reserve Bank of India (Rural Co-operative Bank...
65    Reserve Bank of India (Rural Co-operative Bank...

=== nbfc_standalone ===
Series([], )

=== nbfc_citation_unmatched ===
0     Reserve Bank of India (Credit Information Comp...
1     Reserve Bank of India (Credit Information Comp...
2     Reserve Bank of India (Asset Reconstruction Co...
4     Reserve Bank of India (Asset Reconstruction Co...
42    Reserve Bank of India (All India Financial Ins...
44    Reserve Bank of India (All India Financial Ins...
45    Reserve B

In [10]:
print(circulars["bucket"].value_counts(dropna=False))
print(len(circulars))

bucket
not_nbfc                   530
nbfc_citation_unmatched    134
linked                     102
Name: count, dtype: int64
766


In [11]:
circulars["found_text_lead"] = circulars["text"].apply(
    lambda t: [norm(x) for x in (p1.findall(t[:600]) + p2.findall(t[:600]))] if isinstance(t, str) else []
)
circulars["has_citation_lead"] = circulars["found_title"].apply(len).gt(0) | circulars["found_text_lead"].apply(len).gt(0)

def bucket_v2(row):
    if row["match_method"] != "no_match":
        return "linked"
    if not row["is_nbfc_relevant"]:
        return "not_nbfc"
    if not row["has_citation_lead"]:
        return "nbfc_standalone"
    return "nbfc_citation_unmatched"

circulars["bucket_v2"] = circulars.apply(bucket_v2, axis=1)
print(circulars["bucket_v2"].value_counts())

bucket_v2
not_nbfc                   530
nbfc_citation_unmatched    119
linked                     102
nbfc_standalone             15
Name: count, dtype: int64


In [13]:
pd.set_option('display.max_colwidth', 300)

# does anything here look like it SHOULDN'T be excluded?
print(circulars[circulars["bucket_v2"]=="not_nbfc"]["title"].sample(100, random_state=1).to_string())

106                                                                                                               Reserve Bank of India (Urban Co-operative Banks – Cash Reserve Ratio and Statutory Liquidity Ratio) Directions, 2025 (Updated as on August 25, 2026)
591                                                                                                                                       Reserve Bank of India (Commercial Banks – Cash Reserve Ratio and Statutory Liquidity Ratio) Third Amendment Directions, 2026
419                                                                                                                                                                                                                             NOP-INR position of Authorised Dealers
329                                                                                                                                                             Reserve Bank of India (Rural Co-operative Banks – C

In [14]:
ref_pattern = re.compile(r"([A-Z]+(?:\.[A-Z]+)+\.\d+)/([\d-]+)/(\d{4}-\d{2})")

circulars["subject_code"] = circulars["text"].str.extract(ref_pattern)[1]
master_directions["subject_code"] = master_directions["text"].str.extract(ref_pattern)[1]

print(circulars["subject_code"].notna().mean())       # what % of circulars even have this line
print(master_directions["subject_code"].notna().mean())

0.49477806788511747
0.75


In [15]:
dupe_codes = master_directions.dropna(subset=["subject_code"]).groupby("subject_code")["id"].nunique()
dupe_codes[dupe_codes > 1]

subject_code
33-01-010    30
Name: id, dtype: int64

In [16]:
code_matches = circulars.merge(
    master_directions.dropna(subset=["subject_code"])[["id","subject_code"]],
    on="subject_code", how="left", suffixes=("", "_master")
)

# where code-match succeeds but your text/title match didn't
recovered = code_matches[(code_matches["match_method"] == "no_match") & code_matches["id_master"].notna()]
print(f"{len(recovered)} previously unmatched circulars recoverable via subject_code")

# where both methods matched, do they point to the same master direction?
both = code_matches[(code_matches["match_method"] != "no_match") & code_matches["id_master"].notna()]
disagreements = both[both["matched_id"] != both["id_master"]]
print(f"{len(disagreements)} / {len(both)} disagree between text-match and code-match")

4461 previously unmatched circulars recoverable via subject_code
1133 / 1177 disagree between text-match and code-match
